# 21. How GNFW pressure-profile shape parameters shape the tSZ power spectrum

Companion to nb20: fix the **self-similar amplitude exponents** to
$\alpha=\beta=2/3$ in

$$P_{500}=P_{500,0}\,(M_{500c}/M_\odot)^{\alpha}\,E(z)^{2+\beta},$$

and vary only the **GNFW shape** parameters
$P_0,\,c_{500},\,\alpha_\mathrm{GNFW},\,\beta_\mathrm{GNFW},\,\gamma$
from `SelfSimilarGNFWPressureProfile` (Arnaud et al. 2010 defaults at the pivot).
We plot the **shape** of the halo-model tSZ $D_\ell$ — each curve normalized at
$\ell=2000$, so $P_0$ and $P_{500,0}$ cancel and only the radial-profile-driven
change in the $\ell$-dependence remains.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import sys
sys.path.insert(0, "/scratch/scratch-lxu/flamingo_repo/src")

import numpy as np
import matplotlib.pyplot as plt
import jax.numpy as jnp

from flamingo.catalogue import D3A_COSMOLOGY
from flamingo.profiles.custom_pressure import SelfSimilarGNFWPressureProfile
from hmfast.halos import HaloModel
from hmfast.tracers import tSZTracer

hm = HaloModel(cosmology=D3A_COSMOLOGY)
ell = jnp.logspace(1.3, np.log10(6000), 40)
m_grid = jnp.logspace(11.0, 15.5, 50)
z_grid = jnp.geomspace(0.01, 3.0, 50)
pref = np.asarray(ell) * (np.asarray(ell) + 1) / (2 * np.pi)
ell_np = np.asarray(ell)
ELL_REF = 2000.0   # normalization point for the shape comparison

# Self-similar amplitude exponents (nb17 convention; same as nb20)
ALPHA_SS = BETA_SS = 2.0 / 3.0

# Arnaud A10 / hmfast default GNFW shape (SelfSimilarGNFWPressureProfile defaults)
A10 = dict(P0=8.130, c500=1.156, alpha=1.0620, beta=5.4807, gamma=0.3292)


def tsz_dl(**gnfw_kw):
    """Halo-model tSZ D_ell for fixed self-similar amplitude and given GNFW shape."""
    prof = SelfSimilarGNFWPressureProfile(
        P500_0=1.0, alpha_amp=ALPHA_SS, beta_amp=BETA_SS, **{**A10, **gnfw_kw}
    )
    tr = tSZTracer(profile=prof)
    cl = np.asarray(hm.cl_1h(tr, tr, l=ell, m=m_grid, z=z_grid)) \
       + np.asarray(hm.cl_2h(tr, tr, l=ell, m=m_grid, z=z_grid))
    dl = pref * cl
    return dl / np.interp(ELL_REF, ell_np, dl)   # normalized at ELL_REF


In [ ]:
# One-at-a-time sweeps around the A10 pivot (P0 omitted — it cancels under normalization)
GAMMAS = [0.15, 0.25, A10["gamma"], 0.45, 0.60]          # inner slope
BETAS_GNFW = [4.0, 5.0, A10["beta"], 6.5, 8.0]           # outer slope
C500S = [0.80, 1.00, A10["c500"], 1.40, 1.80]             # concentration in p(x)
ALPHAS_GNFW = [0.80, 1.00, A10["alpha"], 1.20, 1.40]     # transition sharpness

dl_gamma = {g: tsz_dl(gamma=g) for g in GAMMAS}
dl_beta = {b: tsz_dl(beta=b) for b in BETAS_GNFW}
dl_c500 = {c: tsz_dl(c500=c) for c in C500S}
dl_alpha = {a: tsz_dl(alpha=a) for a in ALPHAS_GNFW}

peak = lambda d: int(ell_np[np.argmax(d)])
print("peak ell vs gamma     :", {g: peak(d) for g, d in dl_gamma.items()})
print("peak ell vs beta_GNFW :", {b: peak(d) for b, d in dl_beta.items()})
print("peak ell vs c500      :", {c: peak(d) for c, d in dl_c500.items()})
print("peak ell vs alpha_GNFW:", {a: peak(d) for a, d in dl_alpha.items()})


In [ ]:
def _plot_panel(ax, curves, xlabel, title, default_val, fmt="{:.2f}", tag="A10"):
    cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(curves)))
    for col, (val, dl) in zip(cmap, curves.items()):
        is_default = abs(val - default_val) < 1e-3
        ax.plot(ell_np, dl, color=col, lw=2.6 if is_default else 1.6,
                label=fmt.format(val) + (f" ({tag})" if is_default else ""))
    ax.set_title(title)
    ax.set_xlabel(r"$\ell$")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.axvline(ELL_REF, color="0.7", ls=":", lw=1)
    ax.set_xlim(ell_np.min(), ell_np.max())
    ax.legend(fontsize=7)


fig, axes = plt.subplots(2, 2, figsize=(13, 10), sharey=True)
ss_label = fr"$\alpha=\beta={ALPHA_SS:.2f}$ (self-sim.)"

_plot_panel(axes[0, 0], dl_gamma, r"$\ell$",
            fr"vary inner slope $\gamma$  ({ss_label})", A10["gamma"], fmt=r"$\gamma={:.2f}$")
_plot_panel(axes[0, 1], dl_beta, r"$\ell$",
            fr"vary outer slope $\beta_\mathrm{{GNFW}}$  ({ss_label})", A10["beta"],
            fmt=r"$\beta_\mathrm{{GNFW}}={:.2f}$")
_plot_panel(axes[1, 0], dl_c500, r"$\ell$",
            fr"vary $c_{{500}}$  ({ss_label})", A10["c500"], fmt=r"$c_{{500}}={:.2f}$")
_plot_panel(axes[1, 1], dl_alpha, r"$\ell$",
            fr"vary $\alpha_\mathrm{{GNFW}}$  ({ss_label})", A10["alpha"],
            fmt=r"$\alpha_\mathrm{{GNFW}}={:.2f}$")

axes[0, 0].set_ylabel(r"$D_\ell\,/\,D_{\ell=2000}$ (shape)")
axes[1, 0].set_ylabel(r"$D_\ell\,/\,D_{\ell=2000}$ (shape)")
for ax in axes.ravel():
    ax.set_ylim(3e-2, 6)

fig.suptitle("tSZ power-spectrum shape vs GNFW pressure profile "
             "(self-similar amplitude fixed)", y=1.01)
fig.tight_layout(); plt.show()


### Trends

- **Inner slope $\gamma$** (top left): controls the central pressure cusp.
  Steeper cores ($\gamma\uparrow$) concentrate emission in the inner beam,
  boosting high-$\ell$ power relative to the large-scale tail.
- **Outer slope $\beta_\mathrm{GNFW}$** (top right): sets how quickly pressure
  falls beyond $R_{500c}$. A shallower outer slope keeps more extended emission,
  reddening the spectrum (more low-$\ell$ power).
- **Concentration $c_{500}$** (bottom left): rescales the dimensionless radius
  in $p(x)$; higher $c_{500}$ compresses the profile in physical units for a
  fixed $R_{500c}$, sharpening the projected signal.
- **Transition exponent $\alpha_\mathrm{GNFW}$** (bottom right): governs how
  sharply the profile turns over from the inner power law to the outer slope;
  the effect is subdominant compared to $\gamma$ and $\beta_\mathrm{GNFW}$ but
  still shifts the peak $\ell$ at the few-hundred level.
- $P_0$ is **not** varied: it is an overall multiplicative factor on $P_e(r)$ and
  cancels entirely once each curve is normalized at $\ell=2000$.